# F5 · Business Intelligence — export del modelo estrella a Power BI

Construye el modelo de datos en **estrella** (`dim_team`, `dim_event`, `fact_matches`)
y lo exporta como CSVs a `data/bi/` para abrirlos en **Power BI Desktop**.

El código vive en **`src/bi_export.py`** (pandas + numpy, sin dependencias extra).
Este notebook solo lo importa, ejecuta el export y muestra un resumen.

Entradas (por defecto en `data/`):
- `interim/matches.parquet`
- `interim/rankings.parquet`
- `processed/features.parquet`
- (opcional) `interim/teams.parquet`, `team_profiles.parquet`, `team_stats.parquet`

Salidas (`data/bi/`): `dim_team.csv`, `dim_event.csv`, `fact_matches.csv`.
Diccionario completo: `docs/diccionario_datos.md`.


In [1]:
# Monta Google Drive (solo en Colab) para acceder a los datos y a src/.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # fuera de Colab (local)


Mounted at /content/drive


In [2]:
import sys
import os
from pathlib import Path

import pandas as pd

ROOT = Path("/content/drive/MyDrive/Analisis de datos/Proyecto final")
if not ROOT.exists():
    ROOT = Path.cwd()  # fallback local
os.environ["VALORANT_DATA_DIR"] = str(ROOT / "data")
sys.path.insert(0, str(ROOT / "src"))

import bi_export
from bi_export import export_star_schema, validate_schema

In [3]:
# Construye el modelo estrella (en memoria) sin escribir aún.
tables = export_star_schema()

print("Tablas del modelo estrella:")
for name, df in tables.items():
    print(f"  {name:<14} {len(df):>8,} filas  x  {len(df.columns):>2} columnas")

Tablas del modelo estrella:
  dim_team         20,286 filas  x  22 columnas
  dim_event         2,992 filas  x   5 columnas
  fact_matches    107,242 filas  x  34 columnas
  dim_map          20,124 filas  x   9 columnas


In [4]:
# Vista previa de cada tabla
for name, df in tables.items():
    print(f"\n===== {name} =====")
    display(df.head(3))


===== dim_team =====


,team_id,team_name,team_name_norm,logo,region,country,country_code,country_name,rank,rating,...,record,wins,losses,earnings_usd,total_winnings_usd,roster_size,streak,streak_value,streak_type,en_ranking
0,1,0-12 in parlays,0 12 in parlays,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,<NA>,NaN,<NA>,,0
1,2,0.2,0 2,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,<NA>,NaN,<NA>,,0
2,3,0 FOCO,0 foco,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,...,NaN,<NA>,<NA>,NaN,NaN,<NA>,NaN,<NA>,,0



===== dim_event =====


,event_id,event_name,num_partidos,fecha_min,fecha_max
0,1,#PlayAtHome #1,30,2020-11-09,2020-11-14
1,2,#PlayAtHome #2,63,2020-12-07,2020-12-12
2,3,"$1,000 FreshCut Cup",8,2023-03-11,2023-03-11



===== fact_matches =====


,match_id,team1_id,team2_id,winner_id,event_id,team1,team2,winner,stage,date,...,f_rank_diff,f_form_diff,f_winrate_diff,f_mapscore_diff,f_experience_diff,f_rest_diff,f_streak_diff,f_map_winrate_diff,f_profile_rating_diff,f_winnings_diff
0,190,17795,9129,9129,204,TooGoodForLan,L'institut,team2,Playoffs,"Sat, April 25, 2020",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,191,13094,5975,5975,204,Prodigy,fish123,team2,Playoffs,"Sat, April 25, 2020",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,192,11308,11024,11308,204,Ninjas in Pyjamas,need more DM,team1,Playoffs,"Sat, April 25, 2020",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



===== dim_map =====


,team_id,team_name,map,games,wins,losses,win_pct,atk_rwin_pct,def_rwin_pct
0,1001,team heretics,Abyss,15,4,11,0.27,0.47,0.37
1,1001,team heretics,Ascent,85,56,29,0.66,0.56,0.61
2,1001,team heretics,Bind,64,43,21,0.67,0.58,0.52


In [5]:
# Chequeos de integridad (PK únicas, FKs válidas, sin nulos en FKs)
checks = validate_schema(tables)
for k, v in checks.items():
    print(f"{k:<16} {v}")
assert all(checks.values()), "Fallo de integridad referencial"

dim_team_filas   20286
dim_event_filas  2992
fact_filas       107242
team_id_unicos   True
event_id_unicos  True
match_id_unicos  True
fk_team1_ok      True
fk_team2_ok      True
fk_winner_ok     True
fk_event_ok      True
sin_nulos_fk     True
prob_cob_%       90.91
dim_map_filas    20124
dim_map_sin_nulos_id True


In [6]:
# Exporta los CSVs (UTF-8 con BOM) a data/bi/
paths = bi_export.write_csvs(tables, bi_export.DEFAULT_OUTPUT_DIR)
for p in paths:
    print(f"{p.name:<20} {p.stat().st_size / 1e6:>8.2f} MB")

dim_team.csv             1.17 MB
dim_event.csv            0.19 MB
fact_matches.csv        32.40 MB
dim_map.csv              0.83 MB


## Relaciones a crear en Power BI

| Relación | Desde (FK) | Hacia (PK) | Activa |
|---|---|---|---|
| 1 | `fact_matches.team1_id` | `dim_team.team_id` | No |
| 2 | `fact_matches.team2_id` | `dim_team.team_id` | No |
| 3 | `fact_matches.winner_id` | `dim_team.team_id` | Sí |
| 4 | `fact_matches.event_id` | `dim_event.event_id` | Sí |

`dim_team` es una *role-playing dimension* (tres FKs a la misma tabla). Usa
`USERELATIONSHIP(...)` en DAX para análisis desde el equipo A o el equipo B.

> Ver `docs/diccionario_datos.md` para el diagrama completo del esquema,
> la guía de carga paso a paso y el detalle de cada columna.
